# bike 대여량 예측 회귀 모델링

[실습 프로세스] 
1. 데이터 불러오기 
2. 데이터 탐색 
3. 데이터 전처리 
4. 학습/테스트 데이터 분리 
5. 모델 선택 및 학습 
6. 예측 및 평가

# 데이터 불러오기 

In [36]:
import pandas as pd 

In [37]:
df = pd.read_csv('./data/bike_train.csv')
df.head(3)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32


In [38]:
df.info() # Null 값 없음

<class 'pandas.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  str    
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), str(1)
memory usage: 1020.7 KB


# 데이터 탐색 

In [39]:
import numpy as np

In [40]:
df['count'].describe()

count    10886.000000
mean       191.574132
std        181.144454
min          1.000000
25%         42.000000
50%        145.000000
75%        284.000000
max        977.000000
Name: count, dtype: float64

In [41]:
dt = pd.to_datetime(df["datetime"])
print(dt.min(), dt.max())

2011-01-01 00:00:00 2012-12-19 23:00:00


# 데이터 전처리

## datetime을 분해해서 파생변수 만들기

In [42]:
df2 = df.copy()
df2["datetime"] = pd.to_datetime(df2["datetime"])
df2["year"] = df2["datetime"].dt.year
df2["month"] = df2["datetime"].dt.month
df2["day"] = df2["datetime"].dt.day
df2["hour"] = df2["datetime"].dt.hour
df2["dayofweek"] = df2["datetime"].dt.dayofweek
df2["is_weekend"] = (df2["dayofweek"] >= 5).astype(int) 

## 데이터 누수변수 삭제
> 데이터 누수변수 = 예측시점에 알 수 없어야 할 정보가 훈련데이터셋에 포함되어 모델의 성능을 높인 변수

In [43]:
df2 = df2.drop(columns=["datetime", "casual", "registered"])
df2.head()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,count,year,month,day,hour,dayofweek,is_weekend
0,1,0,0,1,9.84,14.395,81,0.0,16,2011,1,1,0,5,1
1,1,0,0,1,9.02,13.635,80,0.0,40,2011,1,1,1,5,1
2,1,0,0,1,9.02,13.635,80,0.0,32,2011,1,1,2,5,1
3,1,0,0,1,9.84,14.395,75,0.0,13,2011,1,1,3,5,1
4,1,0,0,1,9.84,14.395,75,0.0,1,2011,1,1,4,5,1


## 범주형/수치형 구분 

In [44]:
cat_cols = ["season","holiday","workingday","weather","year","month","hour","dayofweek","is_weekend"]
num_cols = ["temp","atemp","humidity","windspeed"]

# 학습/테스트 데이터 분리

In [45]:
from sklearn.model_selection import train_test_split

X = df2.drop(columns=["count"])
y = df2["count"]

# (옵션1) 랜덤 분할
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# (옵션2: 권장) 시간 기반 분할
split_idx = int(len(df2) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

> 왜 시간 기반 분할을 해야하는가?
이 데이터는 시간 파생변수를 쓰기 때문에 시간 분할이 더 자연스러움

보통 전처리에서 datetime을 year/month/hour/dayofweek로 쪼갬 > 모델은 “2012년의 특정 시간대 패턴”을 학습할 수 있는데, 랜덤 분할이면 테스트에도 같은 연/월/시간대가 섞여 있어서 쉽게 맞추는 평가가 됨

시간기반 분할은 적어도 “뒤쪽 기간(미래)”을 테스트로 두기 때문에,  [2011~2012초반으로 학습 → 2012후반을 예측] 같은 형태로 “미래 예측” 검증이 되어야 함

> 데이터 전체를 학습해서 앞으로의 예측 회귀 모델을 만들면 되는 거 아닌가? 
최종 배포용 모델(전체 데이터로 학습) 설계 전에: 시간 기반으로 한 번은 검증해야 “앞으로 잘 맞출지”를 알 수 있음

# 모델 선택 및 학습

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [47]:
# 훈련 데이터로 ‘학습’이 필요한 변환(스케일링/원핫)을 누수없이 적용하기 위한 전처리 fit 
#  <> 데이터 자체에 대한 전처리
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop"
)

In [48]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}

# 예측 및 평가 

In [49]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

In [54]:
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

# 표로 변환 + 정렬(RMSE 낮은 순)
df_res = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

df_show = df_res.copy()
df_show["MAE"] = df_show["MAE"].round(2)
df_show["RMSE"] = df_show["RMSE"].round(2)
df_show["R2"] = df_show["R2"].round(3)

print("=== Model Comparison (lower RMSE/MAE is better, higher R2 is better) ===")
print(df_show.to_string(index=False))

# 베스트 모델 요약(RMSE 기준)
best = df_res.iloc[0]
print(f"\nBest by RMSE: {best['Model']}  |  MAE={best['MAE']:.2f}, RMSE={best['RMSE']:.2f}, R2={best['R2']:.3f}")

=== Model Comparison (lower RMSE/MAE is better, higher R2 is better) ===
           Model   MAE   RMSE    R2
    RandomForest 62.63  94.04 0.813
GradientBoosting 89.87 124.92 0.670
LinearRegression 98.38 131.27 0.636
           Ridge 98.49 131.40 0.635

Best by RMSE: RandomForest  |  MAE=62.63, RMSE=94.04, R2=0.813


[해석]
- MAE: 평균적으로 얼마나 틀리는지(절대오차 평균). 낮을수록 좋음
  - MAE = 62.63 → 평균적으로 실제 대여량과 약 63 정도 차이
- RMSE: 큰 오차에 더 벌점을 주는 오차(제곱 기반). 낮을수록 좋음
  - RMSE = 94.04 → 큰 오차까지 고려해도 평균적으로 약 94 정도 차이
- R²: 평균만 찍는 것 대비 설명력. 높을수록 좋음
  - R² = 0.813 → 테스트 구간에서 count 변동의 약 81.3%를 설명

> 네 비교군 중에서 RandomForest가 압도적으로 오차도 가장 작고(MAE/RMSE 최소), 설명력도 가장 높음(R²최대)

In [ ]:
# 베스트 대비 성능 차이
df_gap = df_res.copy()
df_gap["RMSE_Δ_vs_best"] = (df_gap["RMSE"] - best["RMSE"]).round(2)
df_gap["R2_Δ_vs_best"] = (df_gap["R2"] - best["R2"]).round(3)
df_gap = df_gap[["Model", "RMSE_Δ_vs_best", "R2_Δ_vs_best"]]
print("=== Gap vs Best (RMSE/R2) ===")
print(df_gap.to_string(index=False))

=== Gap vs Best (RMSE/R2) ===
           Model  RMSE_Δ_vs_best  R2_Δ_vs_best
    RandomForest            0.00         0.000
GradientBoosting           30.88        -0.143
LinearRegression           37.23        -0.177
           Ridge           37.36        -0.178


- RMSE_Δ_vs_best: 1등 대비 RMSE가 얼마나 더 큰지
- R2_Δ_vs_best: 1등 대비 R²가 얼마나 더 낮은지(음수)

> 선형(Linear/Ridge)보다 트리 앙상블이 훨씬 유리했고, 그중에서도 RandomForest가 가장 안정적으로 대여량을 맞춤